# ABSA Resume Training — Tiếp tục từ Checkpoint E15

**Chạy trên Colab mới hoặc tiếp tục Colab cũ — không cần train lại từ đầu.**

| Fix | Nội dung | Tác động |
|-----|----------|----------|
| P1 | `monitor = 0.7×Macro + 0.3×Micro` | Early stopping ưu tiên Macro-F1 |
| P3 | Cosine LR mới với peak `2e-5` | Fine-tune range, không overfit |
| P5 | Threshold tối ưu F1 trực tiếp | Align với metric cuối cùng |

**Yêu cầu:** `best_absa_v3.pt` đã lưu trên Drive tại `KLTN/ketquamohinh/`

**Kết quả E15 hiện tại:** Macro=0.6679 | Micro=0.8727

In [1]:

# !pip install transformers scikit-learn torch --quiet

import gc, os, json, random, warnings
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizerFast,
    RobertaModel,
    get_cosine_schedule_with_warmup,
)
from sklearn.metrics import (
    f1_score, classification_report,
    precision_score, recall_score,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


In [9]:
CKPT_GDRIVE_URL = "https://drive.google.com/file/d/1CiDwxuyB3dwfBjv0OzmUFCPgx4KJSBgW/view?usp=drive_link"

In [10]:
from google.colab import drive
drive.mount("/content/drive")

import gdown, os, shutil

def gdown_id(url, out):
    if os.path.exists(out):
        print(f"  Already exists: {out}")
        return
    file_id = url.split("/d/")[1].split("/")[0]
    gdown.download(f"https://drive.google.com/uc?id={file_id}", out, quiet=False)

gdown_id("https://drive.google.com/file/d/1cV6aTgNeYpODsPy_Nyca1TpgOEolqKrK/view?usp=drive_link", "/content/train.csv")
gdown_id("https://drive.google.com/file/d/1NbNkAusKZ8K8kfYowVcfAEZElrpgXRqv/view?usp=drive_link", "/content/val.csv")
gdown_id("https://drive.google.com/file/d/1DrKbI7CGKbg_ifGzLsWti8uZCD64YLe1/view?usp=drive_link", "/content/test.csv")
gdown_id("https://drive.google.com/file/d/1kqZOjKfsE09w-nNecv4PbVQ_vbt5NPzJ/view?usp=drive_link", "/content/pos_weight.json")



TRAIN_PATH  = "/content/train.csv"
VAL_PATH    = "/content/val.csv"
TEST_PATH   = "/content/test.csv"
POS_W_PATH  = "/content/pos_weight.json"
CKPT_PATH   = "/content/best_absa_v3.pt"      # sẽ download về đây
SAVE_PATH   = "/content/best_absa_resume.pt"   # checkpoint mới lưu vào đây
THRESH_PATH = "/content/best_thresholds_resume.npy"
DRIVE_DIR   = "/content/drive/MyDrive/KLTN/ketquamohinh/"

os.makedirs(DRIVE_DIR, exist_ok=True)

# Download checkpoint từ link bạn cung cấp
if not os.path.exists(CKPT_PATH):
    print("Downloading checkpoint từ Google Drive...")
    gdown_id(CKPT_GDRIVE_URL, CKPT_PATH)
    print(f"  Done: {CKPT_PATH}  ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)")
else:
    print(f"Checkpoint already at: {CKPT_PATH}  ({os.path.getsize(CKPT_PATH)/1e6:.1f} MB)")

LABEL_COLS = [
    "scent_positive",    "scent_negative",
    "longevity_positive","longevity_negative",
    "packaging_positive","packaging_negative",
    "service_positive",  "service_negative",
]
NUM_LABELS = len(LABEL_COLS)
ASPECTS    = ["scent", "longevity", "packaging", "service"]

# ── Resume hyperparams ─────────────────────────────────────
RESUME_EPOCHS  = 10
BATCH_SIZE     = 8
ACCUM_STEPS    = 8
LR_HEAD_RESUME = 2e-5
LR_TOP_RESUME  = 5e-6
LR_DECAY       = 0.85
PATIENCE       = 5

# ── Load pos_weight ────────────────────────────────────────
import json as _json
with open(POS_W_PATH, "r") as _f:
    _pw_dict = _json.load(_f)

import torch
POS_WEIGHT = torch.tensor(
    [max(1.0, _pw_dict[c] ** 0.5) for c in LABEL_COLS],
    dtype=torch.float
).to(device)

print(f"\nNUM_LABELS   : {NUM_LABELS}")
print(f"Resume epochs: {RESUME_EPOCHS} (E16 → E{15+RESUME_EPOCHS})")
print(f"LR head      : {LR_HEAD_RESUME:.1e}")
print(f"LR top layers: {LR_TOP_RESUME:.1e}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  Already exists: /content/train.csv
  Already exists: /content/val.csv
  Already exists: /content/test.csv
  Already exists: /content/pos_weight.json


Downloading...
From (original): https://drive.google.com/uc?id=1CiDwxuyB3dwfBjv0OzmUFCPgx4KJSBgW
From (redirected): https://drive.google.com/uc?id=1CiDwxuyB3dwfBjv0OzmUFCPgx4KJSBgW&confirm=t&uuid=39dd862c-e999-4127-a402-b349454b2efc
To: /content/best_absa_v3.pt
100%|██████████| 502M/502M [00:30<00:00, 16.5MB/s]


  Done: /content/best_absa_v3.pt  (501.9 MB)

NUM_LABELS   : 8
Resume epochs: 10 (E16 → E25)
LR head      : 2.0e-05
LR top layers: 5.0e-06


In [11]:

df_train = pd.read_csv(TRAIN_PATH)
df_val   = pd.read_csv(VAL_PATH)
df_test  = pd.read_csv(TEST_PATH)

print(f"Train : {len(df_train):,} | Val : {len(df_val):,} | Test : {len(df_test):,}")

X_train = df_train["full_text"].tolist()
y_train = df_train[LABEL_COLS].values.astype(float).tolist()
X_val   = df_val["full_text"].tolist()
y_val   = df_val[LABEL_COLS].values.astype(float).tolist()
X_test  = df_test["full_text"].tolist()
y_test  = df_test[LABEL_COLS].values.astype(float).tolist()

tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

# Detect MAX_LEN từ data (nhất quán với v3)
_lengths = [len(tokenizer.encode(t, truncation=False)) for t in X_train[:3000]]
_pct     = np.mean(np.array(_lengths) > 128) * 100
MAX_LEN     = 256 if _pct > 15 else 128
BATCH_SIZE  = 8   if MAX_LEN == 256 else 16
ACCUM_STEPS = 8   if MAX_LEN == 256 else 4

print(f"MAX_LEN={MAX_LEN} | batch={BATCH_SIZE} | accum={ACCUM_STEPS}")

label_array = np.array(y_train)
label_freq  = label_array.sum(axis=0)
label_freq  = np.where(label_freq == 0, 1, label_freq)

Train : 31,010 | Val : 6,652 | Test : 6,660


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

MAX_LEN=256 | batch=8 | accum=8


In [12]:

class ReviewDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = texts, labels
        self.tokenizer, self.max_len = tokenizer, max_len

    def __len__(self): return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.float),
        }


g = torch.Generator(); g.manual_seed(SEED)

train_ds     = ReviewDataset(X_train, y_train, tokenizer, MAX_LEN)
val_ds       = ReviewDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_ds      = ReviewDataset(X_test,  y_test,  tokenizer, MAX_LEN)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          generator=g, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE*2, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train batches : {len(train_loader)} | Val batches : {len(val_loader)}")

Train batches : 3877 | Val batches : 416


In [13]:


class ABSAModel(nn.Module):
    """
    RoBERTa-base + CLS & Attention Pooling → concat(1536) → classifier.
    Giữ nguyên 100% so với v3 để load checkpoint đúng.
    """
    def __init__(self, num_labels: int = 8, dropout: float = 0.2):
        super().__init__()
        self.roberta   = RobertaModel.from_pretrained("roberta-base")
        hidden         = self.roberta.config.hidden_size  # 768
        self.attn_pool = nn.Linear(hidden, 1)
        self.classifier = nn.Sequential(
            nn.LayerNorm(hidden * 2),
            nn.Dropout(dropout),
            nn.Linear(hidden * 2, 512),
            nn.GELU(),
            nn.LayerNorm(512),
            nn.Dropout(dropout / 2),
            nn.Linear(512, num_labels),
        )

    def _attn_pool(self, hidden, mask):
        w = self.attn_pool(hidden).squeeze(-1)
        w = w.masked_fill(mask == 0, float("-inf"))
        w = torch.softmax(w, dim=-1)
        return (hidden * w.unsqueeze(-1)).sum(1)

    def forward(self, input_ids, attention_mask):
        out    = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls    = out.last_hidden_state[:, 0, :]
        attn   = self._attn_pool(out.last_hidden_state, attention_mask)
        pooled = torch.cat([cls, attn], dim=-1)
        return self.classifier(pooled)


model = ABSAModel(num_labels=NUM_LABELS).to(device)
print(f"Model params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model params: 125.4M


In [14]:

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state"])

start_epoch      = ckpt["epoch"]          # = 15
best_val_micro   = ckpt["val_micro"]
best_val_macro   = ckpt["val_macro"]
best_val_monitor = ckpt.get("val_monitor", 0.5*best_val_macro + 0.5*best_val_micro)

# Recompute monitor với công thức MỚI (0.7/0.3) để so sánh đúng
best_score = 0.7 * best_val_macro + 0.3 * best_val_micro

print(f"\n{'='*60}")
print(f"Loaded checkpoint từ Epoch {start_epoch}")
print(f"  Val Micro   = {best_val_micro:.4f}")
print(f"  Val Macro   = {best_val_macro:.4f}")
print(f"  Monitor(v3) = {best_val_monitor:.4f}  (0.5M+0.5m)")
print(f"  Monitor(v4) = {best_score:.4f}  (0.7M+0.3m) ← dùng từ đây")
print(f"{'='*60}")
print(f"\nSẽ train thêm {RESUME_EPOCHS} epochs: E{start_epoch+1} → E{start_epoch+RESUME_EPOCHS}")


Loaded checkpoint từ Epoch 15
  Val Micro   = 0.8727
  Val Macro   = 0.6679
  Monitor(v3) = 0.7703  (0.5M+0.5m)
  Monitor(v4) = 0.7294  (0.7M+0.3m) ← dùng từ đây

Sẽ train thêm 10 epochs: E16 → E25


In [15]:

class FocalLossAdaptive(nn.Module):
    def __init__(self, label_freq, pos_weight,
                 base_gamma=2.0, max_gamma=5.0,
                 base_alpha=0.25, label_smoothing=0.05):
        super().__init__()
        self.ls      = label_smoothing
        freq_norm    = label_freq / label_freq.max()
        gammas       = base_gamma + (max_gamma - base_gamma) * (1 - freq_norm)
        alphas       = np.clip(base_alpha + 0.50 * (1 - freq_norm), 0.25, 0.75)
        self.register_buffer("gammas",     torch.tensor(gammas,     dtype=torch.float))
        self.register_buffer("alphas",     torch.tensor(alphas,     dtype=torch.float))
        self.register_buffer("pos_weight", pos_weight.float())

    def forward(self, logits, targets):
        smooth_t = targets * (1 - self.ls) + 0.5 * self.ls
        probs    = torch.sigmoid(logits)
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, smooth_t, pos_weight=self.pos_weight, reduction="none",
        )
        p_t     = probs * smooth_t + (1 - probs) * (1 - smooth_t)
        alpha_t = self.alphas * smooth_t + (1 - self.alphas) * (1 - smooth_t)
        return (alpha_t * (1 - p_t) ** self.gammas * bce).mean()


criterion = FocalLossAdaptive(label_freq, POS_WEIGHT).to(device)

In [16]:
#
# Resume strategy:
#   - LR_HEAD_RESUME = 2e-5  (v3 dùng 1e-4 lúc đầu)
#   - LR_TOP_RESUME  = 5e-6  (v3 dùng 3e-5 lúc đầu)
#   - 1 cosine cycle mới cho RESUME_EPOCHS epochs
#   - Warmup ngắn 5% (thay vì 10%): model đã warm, không cần nhiều

def get_llrd_params(model, base_lr, decay=LR_DECAY, head_lr=None):
    if head_lr is None:
        head_lr = base_lr * 10
    groups = []
    groups.append({
        "params": list(model.classifier.parameters()) + list(model.attn_pool.parameters()),
        "lr": head_lr, "weight_decay": 0.01, "name": "head",
    })
    layers = model.roberta.encoder.layer
    for i in range(len(layers) - 1, -1, -1):
        groups.append({
            "params": layers[i].parameters(),
            "lr": base_lr * (decay ** (len(layers) - 1 - i)),
            "weight_decay": 0.01, "name": f"layer_{i}",
        })
    groups.append({
        "params": model.roberta.embeddings.parameters(),
        "lr": base_lr * (decay ** len(layers)),
        "weight_decay": 0.01, "name": "embeddings",
    })
    return groups


optimizer    = torch.optim.AdamW(
    get_llrd_params(model, base_lr=LR_TOP_RESUME, head_lr=LR_HEAD_RESUME),
    eps=1e-8
)
total_steps  = (len(train_loader) // ACCUM_STEPS) * RESUME_EPOCHS
warmup_steps = max(1, int(total_steps * 0.05))   # 5% warmup

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
    num_cycles=0.5,
)

scaler = torch.amp.GradScaler("cuda")

print(f"Resume total steps  : {total_steps}")
print(f"Warmup steps        : {warmup_steps} (5%)")
print(f"LR head             : {LR_HEAD_RESUME:.1e}")
print(f"LR top layers       : {LR_TOP_RESUME:.1e}")

Resume total steps  : 4840
Warmup steps        : 242 (5%)
LR head             : 2.0e-05
LR top layers       : 5.0e-06


In [17]:
#
# Thay đổi KEY so với v3:
#   monitor = 0.7*macro + 0.3*micro  (Fix P1)
#   Epoch counter bắt đầu từ start_epoch+1 (= 16)

patience_counter = 0
history          = []

print(f"\n{'='*65}")
print(f"RESUME TRAINING — Starting from Epoch {start_epoch+1}")
print(f"Monitor formula: 0.7×Macro + 0.3×Micro  [FIX P1]")
print(f"Current best: Macro={best_val_macro:.4f} | Micro={best_val_micro:.4f} | Monitor={best_score:.4f}")
print(f"{'='*65}\n")

for local_ep in range(RESUME_EPOCHS):
    epoch = start_epoch + local_ep + 1   # Epoch toàn cục: 16, 17, ...

    # ── Train ─────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)

        with torch.amp.autocast("cuda"):
            loss = criterion(model(ids, mask), lbls) / ACCUM_STEPS

        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if (step + 1) % 300 == 0:
            print(f"  E{epoch} step {step+1}/{len(train_loader)} "
                  f"loss={total_loss/(step+1):.4f} "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")

    optimizer.zero_grad()

    # ── Validate ──────────────────────────────────────────
    model.eval()
    vp_all, vl_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            with torch.amp.autocast("cuda"):
                logits = model(ids, mask)
            vp_all.append((torch.sigmoid(logits).cpu().numpy() > 0.5).astype(int))
            vl_all.append(batch["labels"].numpy())

    vp = np.vstack(vp_all);  vl = np.vstack(vl_all)
    micro    = f1_score(vl, vp, average="micro",  zero_division=0)
    macro    = f1_score(vl, vp, average="macro",  zero_division=0)
    avg_loss = total_loss / len(train_loader)

    # Fix P1: monitor = 0.7*macro + 0.3*micro
    monitor_score = 0.7 * macro + 0.3 * micro

    print(f"\n{'='*65}")
    print(f"Epoch {epoch} (resume +{local_ep+1}/{RESUME_EPOCHS}) | "
          f"loss={avg_loss:.4f} | Micro={micro:.4f} | Macro={macro:.4f} | Monitor={monitor_score:.4f}")
    print(f"{'='*65}")

    history.append({
        "epoch": epoch, "loss": avg_loss,
        "micro": micro, "macro": macro, "monitor": monitor_score,
    })

    if monitor_score > best_score:
        best_score = monitor_score
        patience_counter = 0
        torch.save({
            "epoch":        epoch,
            "model_state":  model.state_dict(),
            "val_micro":    micro,
            "val_macro":    macro,
            "val_monitor":  monitor_score,
            "max_len":      MAX_LEN,
            "label_cols":   LABEL_COLS,
            "resumed_from": start_epoch,
        }, SAVE_PATH)
        # Backup lên Drive ngay
        import shutil
        shutil.copy(SAVE_PATH, f"{DRIVE_DIR}/best_absa_resume_e{epoch}.pt")
        print(f"✅ Saved | Macro={macro:.4f} | Micro={micro:.4f} | Monitor={monitor_score:.4f}\n")
    else:
        patience_counter += 1
        print(f"⚠️  No improvement {patience_counter}/{PATIENCE} "
              f"(best Monitor={best_score:.4f})\n")
        if patience_counter >= PATIENCE:
            print("🛑 Early stopping!")
            break

    gc.collect(); torch.cuda.empty_cache()

# Summary
best_h = max(history, key=lambda h: h["monitor"])
print(f"\n{'='*65}")
print(f"🏆 Resume best: Epoch {best_h['epoch']} | "
      f"Macro={best_h['macro']:.4f} | Micro={best_h['micro']:.4f}")
print(f"\nResume History (E{start_epoch+1}→E{epoch}):")
print(f"  {'E':>3} {'loss':>8} {'micro':>8} {'macro':>8} {'monitor':>9}")
for h in history:
    marker = " ← best" if h["epoch"] == best_h["epoch"] else ""
    print(f"  E{h['epoch']:02d} {h['loss']:>8.4f} {h['micro']:>8.4f} "
          f"{h['macro']:>8.4f} {h['monitor']:>9.4f}{marker}")

print(f"\n--- So sánh ---")
print(f"  Trước resume (E15): Macro=0.6679 | Micro=0.8727")
print(f"  Sau resume  (E{best_h['epoch']:02d}): Macro={best_h['macro']:.4f} | Micro={best_h['micro']:.4f}")


RESUME TRAINING — Starting from Epoch 16
Monitor formula: 0.7×Macro + 0.3×Micro  [FIX P1]
Current best: Macro=0.6679 | Micro=0.8727 | Monitor=0.7294

  E16 step 300/3877 loss=0.0047 lr=3.06e-06
  E16 step 600/3877 loss=0.0046 lr=6.20e-06
  E16 step 900/3877 loss=0.0048 lr=9.26e-06
  E16 step 1200/3877 loss=0.0047 lr=1.24e-05
  E16 step 1500/3877 loss=0.0046 lr=1.55e-05
  E16 step 1800/3877 loss=0.0046 lr=1.86e-05
  E16 step 2100/3877 loss=0.0045 lr=2.00e-05
  E16 step 2400/3877 loss=0.0045 lr=2.00e-05
  E16 step 2700/3877 loss=0.0046 lr=2.00e-05
  E16 step 3000/3877 loss=0.0046 lr=2.00e-05
  E16 step 3300/3877 loss=0.0046 lr=1.99e-05
  E16 step 3600/3877 loss=0.0046 lr=1.99e-05

Epoch 16 (resume +1/10) | loss=0.0046 | Micro=0.8979 | Macro=0.7152 | Monitor=0.7700
✅ Saved | Macro=0.7152 | Micro=0.8979 | Monitor=0.7700

  E17 step 300/3877 loss=0.0043 lr=1.98e-05
  E17 step 600/3877 loss=0.0043 lr=1.98e-05
  E17 step 900/3877 loss=0.0042 lr=1.97e-05
  E17 step 1200/3877 loss=0.0042 lr=1.

In [20]:
# ══════════════════════════════════════════════════════════════
#  RESUME v4 — Warm Restart từ E24 (best checkpoint)
#  LR peak mới: 5e-6  |  Cosine decay  |  10 epochs
# ══════════════════════════════════════════════════════════════
import gc, numpy as np, torch, torch.nn as nn
from sklearn.metrics import f1_score
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# ── Config ────────────────────────────────────────────────────
CKPT_PATH    = f"{DRIVE_DIR}/best_absa_resume_e24.pt"   # đổi nếu tên khác
SAVE_PATH    = f"{DRIVE_DIR}/best_absa_v4.pt"
RESUME_EPOCHS = 10
PATIENCE      = 5
ACCUM_STEPS   = 4          # giữ nguyên như trước
NEW_LR_PEAK   = 5e-6       # thấp hơn lần trước (2e-5), tránh overshoot
T0            = 5           # restart mỗi 5 epoch
T_MULT        = 1           # không tăng chu kỳ

# ── Load checkpoint ───────────────────────────────────────────
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model_state"])
start_epoch   = ckpt["epoch"]          # 24
best_val_macro = ckpt["val_macro"]
best_val_micro = ckpt["val_micro"]
best_score     = ckpt["val_monitor"]

print(f"Loaded E{start_epoch} | Macro={best_val_macro:.4f} | "
      f"Micro={best_val_micro:.4f} | Monitor={best_score:.4f}")

# ── Optimizer + Scheduler mới (warm restart) ─────────────────
optimizer = torch.optim.AdamW(model.parameters(), lr=NEW_LR_PEAK, weight_decay=1e-2)
scheduler = CosineAnnealingWarmRestarts(
    optimizer,
    T_0     = T0 * len(train_loader) // ACCUM_STEPS,  # steps per cycle
    T_mult  = T_MULT,
    eta_min = 1e-8,
)
scaler = torch.amp.GradScaler("cuda")

# ── Training loop ─────────────────────────────────────────────
patience_counter = 0
history          = []

print(f"\n{'='*65}")
print(f"RESUME v4 — Starting from Epoch {start_epoch+1}")
print(f"Warm restart | Peak LR={NEW_LR_PEAK:.0e} | T0={T0} epochs")
print(f"Monitor: 0.7×Macro + 0.3×Micro")
print(f"{'='*65}\n")

for local_ep in range(RESUME_EPOCHS):
    epoch = start_epoch + local_ep + 1   # 25, 26, ...

    # ── Train ─────────────────────────────────────────────────
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbls = batch["labels"].to(device)

        with torch.amp.autocast("cuda"):
            loss = criterion(model(ids, mask), lbls) / ACCUM_STEPS

        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS

        if (step + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        if (step + 1) % 300 == 0:
            print(f"  E{epoch} step {step+1}/{len(train_loader)} "
                  f"loss={total_loss/(step+1):.4f} "
                  f"lr={optimizer.param_groups[0]['lr']:.2e}")

    optimizer.zero_grad()

    # ── Validate ──────────────────────────────────────────────
    model.eval()
    vp_all, vl_all = [], []
    with torch.no_grad():
        for batch in val_loader:
            ids  = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            with torch.amp.autocast("cuda"):
                logits = model(ids, mask)
            vp_all.append((torch.sigmoid(logits).cpu().numpy() > 0.5).astype(int))
            vl_all.append(batch["labels"].numpy())

    vp = np.vstack(vp_all); vl = np.vstack(vl_all)
    micro    = f1_score(vl, vp, average="micro",  zero_division=0)
    macro    = f1_score(vl, vp, average="macro",  zero_division=0)
    avg_loss = total_loss / len(train_loader)
    monitor_score = 0.7 * macro + 0.3 * micro

    print(f"\n{'='*65}")
    print(f"Epoch {epoch} (+{local_ep+1}/{RESUME_EPOCHS}) | "
          f"loss={avg_loss:.4f} | Micro={micro:.4f} | Macro={macro:.4f} | Monitor={monitor_score:.4f}")
    print(f"{'='*65}")

    history.append({"epoch": epoch, "loss": avg_loss,
                    "micro": micro, "macro": macro, "monitor": monitor_score})

    if monitor_score > best_score:
        best_score = monitor_score
        patience_counter = 0
        torch.save({
            "epoch":       epoch,
            "model_state": model.state_dict(),
            "val_micro":   micro,
            "val_macro":   macro,
            "val_monitor": monitor_score,
            "max_len":     MAX_LEN,
            "label_cols":  LABEL_COLS,
            "resumed_from": start_epoch,
        }, SAVE_PATH)
        import shutil
        shutil.copy(SAVE_PATH, f"{DRIVE_DIR}/best_absa_v4_e{epoch}.pt")
        print(f"✅ Saved | Macro={macro:.4f} | Micro={micro:.4f} | Monitor={monitor_score:.4f}\n")
    else:
        patience_counter += 1
        print(f"⚠️  No improvement {patience_counter}/{PATIENCE} "
              f"(best Monitor={best_score:.4f})\n")
        if patience_counter >= PATIENCE:
            print("🛑 Early stopping!")
            break

    gc.collect(); torch.cuda.empty_cache()

# ── Summary ───────────────────────────────────────────────────
best_h = max(history, key=lambda h: h["monitor"])
print(f"\n{'='*65}")
print(f"🏆 v4 best: Epoch {best_h['epoch']} | "
      f"Macro={best_h['macro']:.4f} | Micro={best_h['micro']:.4f} | Monitor={best_h['monitor']:.4f}")
print(f"\nHistory (E{start_epoch+1}→E{epoch}):")
print(f"  {'E':>3} {'loss':>8} {'micro':>8} {'macro':>8} {'monitor':>9}")
for h in history:
    marker = " ← best" if h["epoch"] == best_h["epoch"] else ""
    print(f"  E{h['epoch']:02d} {h['loss']:>8.4f} {h['micro']:>8.4f} "
          f"{h['macro']:>8.4f} {h['monitor']:>9.4f}{marker}")
print(f"\n--- Tổng kết ---")
print(f"  E15 (baseline):  Macro=0.6679 | Micro=0.8727")
print(f"  E24 (v3 best):   Macro=0.7302 | Micro=0.9036")
print(f"  E{best_h['epoch']:02d} (v4 best):   Macro={best_h['macro']:.4f} | Micro={best_h['micro']:.4f}")

Loaded E24 | Macro=0.7302 | Micro=0.9036 | Monitor=0.7822

RESUME v4 — Starting from Epoch 25
Warm restart | Peak LR=5e-06 | T0=5 epochs
Monitor: 0.7×Macro + 0.3×Micro

  E25 step 300/3877 loss=0.0036 lr=5.00e-06
  E25 step 600/3877 loss=0.0038 lr=4.99e-06
  E25 step 900/3877 loss=0.0039 lr=4.97e-06
  E25 step 1200/3877 loss=0.0039 lr=4.95e-06
  E25 step 1500/3877 loss=0.0039 lr=4.93e-06
  E25 step 1800/3877 loss=0.0039 lr=4.89e-06
  E25 step 2100/3877 loss=0.0039 lr=4.86e-06
  E25 step 2400/3877 loss=0.0039 lr=4.81e-06
  E25 step 2700/3877 loss=0.0040 lr=4.76e-06
  E25 step 3000/3877 loss=0.0040 lr=4.71e-06
  E25 step 3300/3877 loss=0.0040 lr=4.65e-06
  E25 step 3600/3877 loss=0.0040 lr=4.59e-06

Epoch 25 (+1/10) | loss=0.0039 | Micro=0.9057 | Macro=0.7351 | Monitor=0.7863
✅ Saved | Macro=0.7351 | Micro=0.9057 | Monitor=0.7863

  E26 step 300/3877 loss=0.0038 lr=4.45e-06
  E26 step 600/3877 loss=0.0038 lr=4.37e-06
  E26 step 900/3877 loss=0.0038 lr=4.29e-06
  E26 step 1200/3877 loss=0

In [21]:

ckpt_best = torch.load(SAVE_PATH, map_location=device)
model.load_state_dict(ckpt_best["model_state"])
model.eval()
print(f"\nLoaded best resume checkpoint | Epoch {ckpt_best['epoch']} | "
      f"Micro={ckpt_best['val_micro']:.4f} | Macro={ckpt_best['val_macro']:.4f}\n")

probs_val, labels_val = [], []
with torch.no_grad():
    for batch in val_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        with torch.amp.autocast("cuda"):
            logits = model(ids, mask)
        probs_val.append(torch.sigmoid(logits).cpu().numpy())
        labels_val.append(batch["labels"].numpy())

probs_val  = np.vstack(probs_val)
labels_val = np.vstack(labels_val)

RARE_SUPPORT    = 200
best_thresholds = np.zeros(NUM_LABELS)

print(f"{'Label':<25} {'T':>6} {'Prec':>7} {'Rec':>7} {'F1':>7} {'Sup':>6}  {'vs 0.5':>7}")
print("-" * 68)

for i, col in enumerate(LABEL_COLS):
    support = int(labels_val[:, i].sum())
    is_rare = support < RARE_SUPPORT
    step    = 0.002 if is_rare else 0.005
    t_range = np.arange(0.05, 0.70, step) if is_rare else np.arange(0.10, 0.91, step)

    baseline_f1 = f1_score(labels_val[:, i], (probs_val[:, i] > 0.5).astype(int), zero_division=0)
    best_t, best_f1 = 0.5, baseline_f1

    for t in t_range:
        preds = (probs_val[:, i] > t).astype(int)
        f1_t  = f1_score(labels_val[:, i], preds, zero_division=0)
        if f1_t > best_f1 + 0.001:   # min_improvement constraint
            best_f1, best_t = f1_t, t

    best_thresholds[i] = best_t
    preds = (probs_val[:, i] > best_t).astype(int)
    p  = precision_score(labels_val[:, i], preds, zero_division=0)
    r  = recall_score(labels_val[:, i],    preds, zero_division=0)
    f1 = f1_score(labels_val[:, i],        preds, zero_division=0)
    delta = f1 - baseline_f1
    print(f"{col:<25} {best_t:>6.3f} {p:>7.3f} {r:>7.3f} {f1:>7.4f} {support:>6}  "
          f"{delta:>+7.4f}" + (" ← rare" if is_rare else ""))

np.save(THRESH_PATH, best_thresholds)
import shutil
shutil.copy(THRESH_PATH, f"{DRIVE_DIR}/best_thresholds_resume.npy")

pf = (probs_val > 0.5).astype(int)
pt = (probs_val > best_thresholds).astype(int)
print(f"\n{'Metric':<20} {'Fixed(0.5)':>12} {'Tuned':>12} {'Delta':>8}")
print("-" * 55)
for name, avg in [("Micro-F1", "micro"), ("Macro-F1", "macro")]:
    f_f = f1_score(labels_val, pf, average=avg, zero_division=0)
    f_t = f1_score(labels_val, pt, average=avg, zero_division=0)
    print(f"{name:<20} {f_f:>12.4f} {f_t:>12.4f} {f_t-f_f:>+8.4f}")


Loaded best resume checkpoint | Epoch 30 | Micro=0.9140 | Macro=0.7610

Label                          T    Prec     Rec      F1    Sup   vs 0.5
--------------------------------------------------------------------
scent_positive             0.500   0.988   0.986  0.9871   6075  +0.0000
scent_negative             0.580   0.762   0.852  0.8046    331  +0.0353
longevity_positive         0.540   0.939   0.922  0.9301   2490  +0.0056
longevity_negative         0.605   0.757   0.794  0.7755    287  +0.1388
packaging_positive         0.590   0.918   0.895  0.9063   1604  +0.0314
packaging_negative         0.566   0.669   0.711  0.6894    128  +0.1044 ← rare
service_positive           0.590   0.617   0.630  0.6237    238  +0.1230
service_negative           0.586   0.919   0.837  0.8760    135  +0.0657 ← rare

Metric                 Fixed(0.5)        Tuned    Delta
-------------------------------------------------------
Micro-F1                   0.9140       0.9396  +0.0256
Macro-F1          

In [22]:

from datetime import datetime

model.eval()
probs_test, labels_test = [], []
with torch.no_grad():
    for batch in test_loader:
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        with torch.amp.autocast("cuda"):
            logits = model(ids, mask)
        probs_test.append(torch.sigmoid(logits).cpu().numpy())
        labels_test.append(batch["labels"].numpy())

probs_test  = np.vstack(probs_test)
labels_test = np.vstack(labels_test)

preds_fixed = (probs_test > 0.5).astype(int)
preds_tuned = (probs_test > best_thresholds).astype(int)

micro_f = f1_score(labels_test, preds_fixed, average="micro", zero_division=0)
macro_f = f1_score(labels_test, preds_fixed, average="macro", zero_division=0)
micro_t = f1_score(labels_test, preds_tuned, average="micro", zero_division=0)
macro_t = f1_score(labels_test, preds_tuned, average="macro", zero_division=0)

print(f"\n{'='*65}")
print(f"KẾT QUẢ CUỐI CÙNG — TEST SET")
print(f"{'='*65}")
print(f"{'Config':<35} {'Micro-F1':>10} {'Macro-F1':>10}")
print(f"{'-'*65}")
print(f"{'Threshold = 0.5':<35} {micro_f:>10.4f} {macro_f:>10.4f}")
print(f"{'Per-label tuned':<35} {micro_t:>10.4f} {macro_t:>10.4f}")
print(f"{'='*65}")
print(f"\nSo sánh:")
print(f"  v3 E15 (val):      Macro=0.6679 | Micro=0.8727")
print(f"  Resume (test):     Macro={macro_t:.4f} | Micro={micro_t:.4f}")

print("\n=== Classification Report (Tuned Thresholds) ===\n")
print(classification_report(labels_test, preds_tuned,
                             target_names=LABEL_COLS, zero_division=0))

ts = datetime.now().strftime("%Y%m%d_%H%M")
pd.DataFrame(
    classification_report(labels_test, preds_tuned, target_names=LABEL_COLS,
                          zero_division=0, output_dict=True)
).T.round(4).to_csv(f"{DRIVE_DIR}/test_report_resume_{ts}.csv")

with open(f"{DRIVE_DIR}/summary_resume_{ts}.txt", "w") as f:
    f.write(f"ABSA Resume từ v3 E15 | {datetime.now()}\n")
    f.write(f"Fix: monitor=0.7M+0.3m, cosine LR, F1 threshold\n")
    f.write(f"Resume epoch: {ckpt_best['epoch']} | Val Micro={ckpt_best['val_micro']:.4f} | Macro={ckpt_best['val_macro']:.4f}\n\n")
    f.write(f"Test (fixed 0.5): Micro={micro_f:.4f} | Macro={macro_f:.4f}\n")
    f.write(f"Test (tuned)    : Micro={micro_t:.4f} | Macro={macro_t:.4f}\n\n")
    f.write(classification_report(labels_test, preds_tuned,
                                   target_names=LABEL_COLS, zero_division=0))

print(f"\n✅ Tất cả đã lưu vào {DRIVE_DIR}")


KẾT QUẢ CUỐI CÙNG — TEST SET
Config                                Micro-F1   Macro-F1
-----------------------------------------------------------------
Threshold = 0.5                         0.9101     0.7502
Per-label tuned                         0.9352     0.8018

So sánh:
  v3 E15 (val):      Macro=0.6679 | Micro=0.8727
  Resume (test):     Macro=0.8018 | Micro=0.9352

=== Classification Report (Tuned Thresholds) ===

                    precision    recall  f1-score   support

    scent_positive       0.99      0.99      0.99      6074
    scent_negative       0.76      0.83      0.79       331
longevity_positive       0.95      0.91      0.93      2490
longevity_negative       0.81      0.84      0.82       287
packaging_positive       0.92      0.88      0.90      1604
packaging_negative       0.64      0.72      0.68       128
  service_positive       0.50      0.52      0.51       238
  service_negative       0.80      0.80      0.80       134

         micro avg       0.94